# Risk scoring notebook

Notebook 01 produced `features.csv` — twenty measurements per vessel. 
This notebook turns those measurements into **one score and one tier per vessel**, and also notes which dimension
each point came from.

## Imp:
1. Nothing is predicted here in the statistical sense.
    - "predicted in statistical sense" => That means a model that learned from outcomes (train data).
    - Learning "what predicts detention" from 2 detentions is not possible in our case
2. What is built instead is a **transparent weighted index**:
    - I decided the factors along with their "weights".
    - E.g. Repetition is worth 20 points because a fault recorded twice means the fix never happened.
    - "Weighted" = each factor gets a fixed number of points
    - It's like a scorecard, not a forecast.

### 0. Setup

In [9]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# feature_builder.py lives at the project root; making it importable from notebooks/
for _dir in (Path.cwd(), *Path.cwd().parents):
    if (_dir / 'pyproject.toml').exists():
        sys.path.insert(0, str(_dir))
        break

from feature_builder import (
    build_features, inspection_features, defs_dated, inspections,
    VESSEL_IDS, PROJECT_ROOT, REFERENCE_DATE,
)
from config import DIMENSIONS

# The feature definitions live in `feature_builder.py`
features = build_features(as_of_date=REFERENCE_DATE, reference_date=REFERENCE_DATE)
print(f'{features.shape[0]} vessels x {features.shape[1]} columns   reference date {REFERENCE_DATE.date()}')




20 vessels x 26 columns   reference date 2026-08-24


## 1. Choosing the target to measure against (yardstick)

We've 20 features, where some will be helpful and some not. To find this out, I need to check against something bad:
- **Detentions**: Has only 2 events. With two positives, any correlation is a comparison of two vessels.
- **Deficiency counts**: a deficiency is a note on a report, a detention stops the ship from sailing. But there are 286 of them. If a feature moves with deficiency counts across 20 vessels, that's a pattern one can trust.



In [3]:
from scipy.stats import spearmanr

yardstick = features.n_deficiencies

candidates = ['repeat_count', 'max_repeat', 'concentration', 'trend', 'open_deficiencies',
              'overdue_maintenance', 'open_audit', 'overdue_audit',
              'known_issues_unresolved', 'known_issue_occurrences', 'equipment_repeats',
              'recent_joiners', 'avg_experience_months',
              'equip_failures_since_inspection', 'equip_repeats_since_inspection',
              'oldest_open_deficiency_days', 'open_from_earlier_inspections',
              'days_since_last_inspection', 'vessel_age']


rows = []
for c in candidates:
    # spearmanr answers one question: do these two things move up and down together
    r_all, p_all = spearmanr(features[c], yardstick)

    # every vessel except V001
    # Because V001 is extreme on many features at the same time - 
    #   most repeats, most open deficiencies, most overdue maintenance, worst trend.
    keep = features.index != 'V001'
    sub = features.loc[keep, c]
    r_ex = np.nan if sub.nunique() < 2 else spearmanr(sub, yardstick[keep]).statistic
    rows.append({'feature': c, 'rho': round(r_all, 2), 'p': round(p_all, 3),
                     'rho_excl_V001': round(r_ex, 2)})

print(pd.DataFrame(rows).sort_values('rho', key=abs, ascending=False).to_string(index=False))


                        feature   rho     p  rho_excl_V001
                     max_repeat  0.65 0.002           0.59
              open_deficiencies  0.56 0.010           0.48
        known_issue_occurrences  0.55 0.013           0.47
                   repeat_count  0.51 0.021           0.43
                          trend  0.47 0.035           0.38
    oldest_open_deficiency_days  0.47 0.036           0.38
        known_issues_unresolved  0.46 0.042           0.40
            overdue_maintenance  0.43 0.062           0.33
  open_from_earlier_inspections  0.38 0.095            NaN
                     open_audit  0.32 0.176           0.26
          avg_experience_months  0.32 0.169           0.29
     days_since_last_inspection  0.31 0.183           0.43
                     vessel_age  0.28 0.239           0.17
                  overdue_audit -0.27 0.245          -0.23
equip_failures_since_inspection -0.22 0.361          -0.08
                  concentration -0.18 0.457          -0.

### Reading above table's columns
- Each feature is tested against the yardstick — total deficiencies — to see whether it carries any fleet-wide signal.
- "rho" is rank correlation, from −1 to +1. It asks whether the two things rise and fall together across the fleet
    - +1 means the feature ranks vessels exactly as the outcome does
    - 0 means it tells you nothing
    - −1 means it ranks them backwards
- "p" is how easily a result this strong could appear by chance.  
    - so p =~ 0.002 is worth believing and p ≈ 0.5 is indistinguishable from noise. 
    - Below 0.05 is the usual bar.
- "rho_excl_V001" repeats the test with V001 removed

### What table shows:
- Seven features clear p < 0.05, and all seven hold up with V001 removed
- They divide into two mechanisms:
    - repetition: 
        - `max_repeat` (0.65)
        - `repeat_count` (0.51)
    - unresolved backlog: 
        - `open_deficiencies` (0.56)
        - `known_issue_occurrences` (0.55)
        - `known_issues_unresolved` (0.46)
        - `oldest_open_deficiency_days` (0.47)

### Imp:
- Features like (`recent_joiners`, `equipment_repeats`) show no measurable relationship with historical deficiency counts.
- That is expected rather than disqualifying: they describe the vessel's current condition, while the deficiency record ends in July 2025
- These features will earn their place further later in my work.


## 2. Which features score, and which only explain

- A weighted sum treats every input as if it earned its place but not every feature belongs in a weighted sum.
- E.g. `open_from_earlier_inspections` is non-zero for exactly one vessel. 
    - In a weighted sum across 20 vessels that is dead weight. 
    - But it is the single most damning fact in the dataset:
    - 3 findings raised, left open, and still outstanding when the next inspector boarded
- Such features go to evidence layer

In [4]:
SCORING_FEATURES = ['repeat_count', 'max_repeat', 'concentration', 'trend',
                    'open_deficiencies', 'known_issue_occurrences', 'known_issues_unresolved',
                    'overdue_audit', 'overdue_maintenance', 'recent_joiners',
                    'avg_experience_months', 'equipment_repeats', 'equip_repeats_since_inspection']

EVIDENCE_ONLY = ['oldest_open_deficiency_days', 'open_from_earlier_inspections',
                 'days_since_last_inspection', 'equip_failures_since_inspection',
                 'open_audit', 'n_deficiencies', 'n_inspections', 'top_category', 'vessel_age']

spread = pd.DataFrame({
    'min': features[SCORING_FEATURES + ['days_since_last_inspection', 'oldest_open_deficiency_days']].min(),
    'max': features[SCORING_FEATURES + ['days_since_last_inspection', 'oldest_open_deficiency_days']].max(),
})
spread['range_over_mean'] = ((spread['max'] - spread['min']) / features[spread.index].mean()).round(2)
spread['scored'] = [i in SCORING_FEATURES for i in spread.index]
print(spread.to_string())

                                    min      max  range_over_mean  scored
repeat_count                      1.000   14.000             3.33    True
max_repeat                        2.000    9.000             2.46    True
concentration                     0.214    0.722             1.55    True
trend                            -3.000    7.000             9.52    True
open_deficiencies                 0.000   10.000             4.26    True
known_issue_occurrences           0.000   20.000             5.48    True
known_issues_unresolved           0.000    4.000             2.16    True
overdue_audit                     0.000    7.000             3.68    True
overdue_maintenance               3.000   11.000             1.32    True
recent_joiners                    0.000    6.000             5.00    True
avg_experience_months           125.100  197.500             0.46    True
equipment_repeats                 1.000    6.000             1.49    True
equip_repeats_since_inspection    1.00

### Reading above table

- `min` and `max` are the lowest and highest values any vessel has for that feature.
- `range_over_mean` 
    - how much this feature actually varies across the fleet.
    - A high number means the vessels genuinely differ.
    - `known_issue_occurrences` at 5.48 runs from 0 to 20 — some vessels have none of this problem and one has a great deal. 
    - A low number means they are all much the same.

## 3. Normalisation

- The features are in incompatible units — `repeat_count` is a count up to 14, `concentration`
is a ratio between 0.21 and 0.72, `avg_experience_months` is in the hundreds. They cannot be
added as they stand.

- **Min–max to 0–1** is used

Two features need handling before scaling:

- **`trend`**: 
    - trend= deficiencies at the last inspection minus deficiencies at the first.
    - V001: +7 => got seven findings worse
    - V005: −3 => got three findings better
    - runs −3 to +7. 
    - trend is clipped at zero before scaling. Otherwise the scale's zero point would sit at the most-improved vessel, so an unchanged vessel would earn risk credit for standing still
    
- **`avg_experience_months`** is inverted — high experience is low risk, and every other
  feature runs the other way.

In [7]:
def minmax(s):
    '''Scale to 0-1. A constant column returns all zeros rather than dividing by zero.'''
    rng = s.max() - s.min()
    if rng == 0:
        return pd.Series(0.0, index=s.index)
    return (s - s.min()) / rng


prepared = features.copy()
prepared['trend_clipped']   = prepared['trend'].clip(lower=0)
prepared['experience_inv']  = -prepared['avg_experience_months']

norm = pd.DataFrame({c: minmax(prepared[c]) for c in
                     ['repeat_count', 'max_repeat', 'concentration', 'trend_clipped',
                      'open_deficiencies', 'known_issue_occurrences', 'known_issues_unresolved',
                      'overdue_audit', 'overdue_maintenance', 'recent_joiners',
                      'experience_inv', 'equipment_repeats', 'equip_repeats_since_inspection']})

print(norm.round(2).to_string())

           repeat_count  max_repeat  concentration  trend_clipped  open_deficiencies  known_issue_occurrences  known_issues_unresolved  overdue_audit  overdue_maintenance  recent_joiners  experience_inv  equipment_repeats  equip_repeats_since_inspection
vessel_id                                                                                                                                                                                                                                                    
V001               1.00        1.00           0.56           1.00                1.0                     1.00                     0.75           0.14                 1.00            1.00            0.42                0.4                            0.00
V002               0.23        0.00           0.14           0.14                0.3                     0.05                     0.25           0.43                 0.12            0.00            0.95                0.6                 

- Every number is now on a 0–1 scale where 0 = the best vessel in the fleet on that feature, 1 = the worst.


- Read these two together and we get the distinction the whole scoring design rests on:

| | V001 | V004 |
|---|---|---|
| `max_repeat` | 1.00 | 0.29 |
| `concentration` | 0.56 | 1.00 |


- V001 is one thing broken over and over
- V004 is many different things wrong in one area

## 4. Dimensions and weights

- The thirteen scoring features collapse into the seven dimensions from `decision_2`. 
- Each dimension is a weighted average of its features on the 0–1 scale, multiplied by its weight.
- The weights sum to 100, so the score reads 0–100.

**The weights cannot be learned** — fitting them needs an outcome with enough events, and there
are two detentions. 

| Dimension | Weight | Reasoning |
|---|---|---|
| Repetition | 20 | the same fault recorded again means the fix never happened |
| Trend | 20 | a worsening vessel is a different problem from a bad one |
| Unresolved | 20 | backlog an inspector can verify, plus issues already known internally |
| Maintenance | 15 | strongest operational signal (ρ +0.55 vs trend) |
| Concentration | 15 | findings clustered in one area point at a system, not bad luck |
| Crew | 5 | experience near-uniform here; weak, kept for structure |
| Equipment | 5 | repeat failures only; raw failure count correlates with nothing |

**Roughly 75 points to inspection behaviour, 25 to current condition** — following the evidence,
since inspection-derived features are the ones with measurable relationships here.

In [10]:
# DIMENSIONS is present in root_path/config.py to avoid hard-coding

print(f"Sum of dimension weights is {sum(d['weight'] for d in DIMENSIONS.values())} ")

def score_fleet(normalised, dimensions=DIMENSIONS):
    '''Return per-dimension point contributions and the total score for every vessel.'''
    out = {}
    for name, cfg in dimensions.items():
        sub_weights = cfg['features']
        dim01 = sum(normalised[f] * w for f, w in sub_weights.items()) / sum(sub_weights.values())
        out[name] = dim01 * cfg['weight']
    contributions = pd.DataFrame(out)
    contributions['score'] = contributions.sum(axis=1)
    return contributions


contributions = score_fleet(norm)
print(contributions.round(1).sort_values('score', ascending=False).to_string())

Sum of dimension weights is 100 
           Repetition  Trend  Unresolved  Maintenance  Concentration  Crew  Equipment  score
vessel_id                                                                                   
V001             20.0   20.0        17.5         15.0            8.4   3.5        1.0   85.5
V004              9.7   14.3        11.2          7.5           15.0   1.1        1.8   60.6
V019              3.9    5.7         4.0          9.4            1.6   2.6        5.0   32.2
V007              6.0    2.9         0.9          9.4            4.2   1.5        4.5   29.4
V013              4.8    5.7         7.1          3.8            1.6   0.8        1.3   25.1
V016              0.9    5.7         3.6          7.5            2.5   2.5        2.3   25.1
V014              1.8    5.7         3.4          9.4            0.0   1.4        3.2   25.0
V011              3.0    5.7         5.7          0.0            5.0   2.7        2.7   24.8
V010              2.8    5.7         

In [12]:
HIGH_CUT, MEDIUM_CUT, SATURATION, MAX_DRIVERS = 55, 28, 0.90, 4
# HIGH_CUT = 55 — score at or above this is tier High
# MEDIUM_CUT = 28 — score at or above this is Medium, below is Low.
# SATURATION = 0.90 — a dimension counts as "saturated" at 90% or more of its own maximum. 
#   E.g. V019 scores 5.0 on Equipment, which is capped at 5, so 5.0 / 5 = 1.00 → flagged.
# MAX_DRIVERS = 4 — never list more than four drivers per vessel.

dims   = list(DIMENSIONS)
caps   = pd.Series({k: v['weight'] for k, v in DIMENSIONS.items()})
points = contributions[dims]

# a dimension is a "driver" if the vessel is above the fleet's 75th percentile on it
elevated  = points > points.quantile(0.75)

# ...and "saturated" if it is near its own maximum, however small that maximum is
saturated = points / caps >= SATURATION


def top_flagged(row, flags, cap=None):
    """Dimension names where `flags` is True for this vessel, highest points first."""
    hits = row[flags.loc[row.name]].sort_values(ascending=False).index.tolist()
    return hits[:cap]


results = contributions.round(1)
results['rank'] = results.score.rank(ascending=False, method='min').astype(int)
results['tier'] = pd.cut(results.score, [-np.inf, MEDIUM_CUT, HIGH_CUT, np.inf],
                         labels=['Low', 'Medium', 'High'], right=False)

results['drivers']   = points.apply(top_flagged, axis=1, flags=elevated, cap=MAX_DRIVERS)
results['saturated'] = points.apply(top_flagged, axis=1, flags=saturated)
results['n_drivers'] = results.drivers.str.len()


results = results.sort_values('score', ascending=False)

print(results.tier.value_counts().reindex(['High', 'Medium', 'Low']).to_string(), '\n')
show = results[['rank', 'score', 'tier']].copy()
for c in ['drivers', 'saturated']:
    show[c] = results[c].str.join(', ').replace('', '-')
print(show.to_string())

tier
High       2
Medium     2
Low       16 

           rank  score    tier                                       drivers                       saturated
vessel_id                                                                                                   
V001          1   85.5    High    Repetition, Trend, Unresolved, Maintenance  Repetition, Trend, Maintenance
V004          2   60.6    High  Concentration, Trend, Unresolved, Repetition                   Concentration
V019          3   32.2  Medium      Maintenance, Equipment, Repetition, Crew                       Equipment
V007          4   29.4  Medium            Maintenance, Repetition, Equipment                       Equipment
V013          5   25.1     Low                        Unresolved, Repetition                               -
V016          5   25.1     Low                                             -                               -
V014          7   25.0     Low                        Maintenance, Equipment      

In [15]:
out = features[['vessel_name', 'vessel_type', 'flag_state', 'vessel_age', 'top_category']].join(results)

# lists are flattened to pipe-separated strings so the CSV round-trips cleanly;
# notebook 03 splits them back on '|'
out['drivers']   = results['drivers'].apply('|'.join)
out['saturated'] = results['saturated'].apply('|'.join)

out = out[['vessel_name', 'vessel_type', 'flag_state', 'vessel_age', 'rank', 'score', 'tier',
           'drivers', 'n_drivers', 'saturated'] + dims].sort_values('rank')
out.to_csv(PROJECT_ROOT / "generated_csvs" / 'scores.csv')

norm.to_csv(PROJECT_ROOT / "generated_csvs" / 'normalised_features.csv')

print(f'scores.csv               {out.shape[0]} rows x {out.shape[1]} cols')
print(f'normalised_features.csv  {norm.shape[0]} rows x {norm.shape[1]} cols')
print()
print(out.head(6).round(1).to_string())

scores.csv               20 rows x 17 cols
normalised_features.csv  20 rows x 13 cols

          vessel_name      vessel_type        flag_state  vessel_age  rank  score    tier                                    drivers  n_drivers                     saturated  Repetition  Trend  Unresolved  Maintenance  Concentration  Crew  Equipment
vessel_id                                                                                                                                                                                                                                                
V001         MV Alpha     Bulk Carrier            Panama          18     1   85.5    High    Repetition|Trend|Unresolved|Maintenance          4  Repetition|Trend|Maintenance        20.0   20.0        17.5         15.0            8.4   3.5        1.0
V004         MV Delta  Chemical Tanker  Marshall Islands          15     2   60.6    High  Concentration|Trend|Unresolved|Repetition          4                 Con